# Q-Commerce Dark Store — SQL Analytics (2015–2025)
## Python + MySQL Integration | 1 Crore Orders · 32 Business Queries

> Companion notebook to `DarkStore_QCommerce_EDA_Project.ipynb`.
> This notebook connects Python to MySQL and runs SQL queries directly —
> results render as pandas DataFrames inline, nothing needs a separate SQL client.

**Pipeline:** CSVs → concatenated in pandas → **one bulk MySQL import** (Phase 1, run once) → year-wise SQL analytics queries (Phases 3-6) → results flow into the EDA notebook by querying MySQL directly (see note below).

### Query Index
| Level | Queries | New in this version |
|---|---|---|
| Basic (Q01–Q10) | Revenue, orders, cities, cancellations, payment mix | — |
| Intermediate (Q11–Q20) | Growth, market share, rolling averages, rankings | — |
| Advanced (Q21–Q30) | RANK/DENSE_RANK/NTILE, cohorts, SLA, Dunzo collapse | — |
| **Validation (Q31–Q32)** | **City-tier + COVID fixes, RFM columns** | Uses `customer_lifetime_orders`, `days_since_last_order`, `promo_code_used`, `estimated_delivery_distance_km` |
---

## Phase 1 : Load Each Year into its OWN MySQL Table (one-time setup)
> Instead of one giant combined table, every year gets its **own table**:
> `qcommerce_orders_2015`, `qcommerce_orders_2016`, ... `qcommerce_orders_2025`.
> Run `SHOW TABLES;` afterwards and all 11 will be listed individually.
> A `qcommerce_orders_all` **VIEW** is also created — a UNION of all 11 tables — so
> "all years combined" queries still work with one simple `FROM qcommerce_orders_all`.

In [1]:
# ── Run this ONCE to load all years into MySQL as separate year-wise tables ──
# pip install pandas sqlalchemy pymysql

import pandas as pd, os, time
from sqlalchemy import create_engine, text
import warnings, os
warnings.filterwarnings('ignore')

MYSQL_USER, MYSQL_PASSWORD = "root", "shubham75"
MYSQL_HOST, MYSQL_PORT     = "127.0.0.1", 3306
MYSQL_DATABASE             = "qcommerce_db"
DATA_PATH = r"D:\Imarticus Learning- PGA 45\GitHub Projects\E-commerce Darkstore"   # ← folder with all 11 CSVs

engine_no_db = create_engine(f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/")
with engine_no_db.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {MYSQL_DATABASE}"))
    conn.commit()

engine = create_engine(f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}")
print(f"Connected → {MYSQL_DATABASE}")

years = list(range(2015, 2026))
table_names = []

for year in years:
    t0 = time.time()

    fpath = os.path.join(DATA_PATH, f"qcommerce_orders_{year}.csv")
    df_year = pd.read_csv(fpath, low_memory=False)

    table_name = f"qcommerce_orders_{year}"
    df_year.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10000)
    table_names.append(table_name)

    print(f"{table_name}: {len(df_year):,} rows written in {time.time()-t0:.0f}s")
    del df_year

print(f"\n {len(table_names)} year-wise tables created")

Connected → qcommerce_db
qcommerce_orders_2015: 100,000 rows written in 14s
qcommerce_orders_2016: 150,000 rows written in 19s
qcommerce_orders_2017: 200,000 rows written in 26s
qcommerce_orders_2018: 250,000 rows written in 32s
qcommerce_orders_2019: 350,000 rows written in 45s
qcommerce_orders_2020: 550,000 rows written in 70s
qcommerce_orders_2021: 1,000,000 rows written in 132s
qcommerce_orders_2022: 1,400,000 rows written in 182s
qcommerce_orders_2023: 1,600,000 rows written in 208s
qcommerce_orders_2024: 1,900,000 rows written in 248s
qcommerce_orders_2025: 2,500,000 rows written in 331s

 11 year-wise tables created


In [2]:
# Create a UNION VIEW that combines all 11 year-wise tables ──
# This lets "all years" queries stay simple (FROM qcommerce_orders_all)
# while each individual year's table remains separately visible in MySQL

union_sql = " UNION ALL ".join([f"SELECT * FROM qcommerce_orders_{y}" for y in years])
create_view_sql = f"CREATE OR REPLACE VIEW qcommerce_orders_all AS {union_sql}"

with engine.connect() as conn:
    conn.execute(text(create_view_sql))
    conn.commit()

print("View created: qcommerce_orders_all (UNION of all 11 year tables)")

View created: qcommerce_orders_all (UNION of all 11 year tables)


In [3]:
# Verify: list ALL year-wise tables + row counts (proves all 11 are visible) ──
with engine.connect() as conn:
    tables = pd.read_sql(text("SHOW TABLES LIKE 'qcommerce_orders_%'"), conn)
print("Tables now visible in MySQL:")
print(tables.to_string(index=False))

print()
print("Row count per year-wise table:")
for year in years:
    with engine.connect() as conn:
        cnt = conn.execute(text(f"SELECT COUNT(*) FROM qcommerce_orders_{year}")).fetchone()[0]
    print(f"qcommerce_orders_{year}: {cnt:,} rows")

with engine.connect() as conn:
    total_view = conn.execute(text("SELECT COUNT(*) FROM qcommerce_orders_all")).fetchone()[0]
print(f"\n TOTAL via view (qcommerce_orders_all): {total_view:,}  (expected 10,000,000)")

Tables now visible in MySQL:
Tables_in_qcommerce_db (qcommerce_orders_%)
                      qcommerce_orders_2015
                      qcommerce_orders_2016
                      qcommerce_orders_2017
                      qcommerce_orders_2018
                      qcommerce_orders_2019
                      qcommerce_orders_2020
                      qcommerce_orders_2021
                      qcommerce_orders_2022
                      qcommerce_orders_2023
                      qcommerce_orders_2024
                      qcommerce_orders_2025
                       qcommerce_orders_all

Row count per year-wise table:
qcommerce_orders_2015: 100,000 rows
qcommerce_orders_2016: 150,000 rows
qcommerce_orders_2017: 200,000 rows
qcommerce_orders_2018: 250,000 rows
qcommerce_orders_2019: 350,000 rows
qcommerce_orders_2020: 550,000 rows
qcommerce_orders_2021: 1,000,000 rows
qcommerce_orders_2022: 1,400,000 rows
qcommerce_orders_2023: 1,600,000 rows
qcommerce_orders_2024: 1,900,000 rows

## Phase 2 : Connect & Verify (run this every session)
> Reconnects to MySQL and confirms all 11 year-wise tables + the `qcommerce_orders_all` view are present.

In [4]:
# Connect to MySQL Server

from sqlalchemy import create_engine, text
MYSQL_USER, MYSQL_PASSWORD = "root", "shubham75"
MYSQL_HOST, MYSQL_PORT     = "127.0.0.1", 3306
MYSQL_DATABASE             = "qcommerce_db"

engine = create_engine(f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}")

def run_query(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

os.makedirs('outputs', exist_ok=True)

# Confirm all year-wise tables are visible
tables = run_query("SHOW TABLES LIKE 'qcommerce_orders_%'")
print(f'Tables visible: {len(tables)}  (expect 11 year tables + 1 view = 12)')
print(tables.to_string(index=False))

total = run_query('SELECT COUNT(*) AS total_rows FROM qcommerce_orders_all')
print(f"\nTotal rows via view: {total['total_rows'][0]:,}")

Tables visible: 12  (expect 11 year tables + 1 view = 12)
Tables_in_qcommerce_db (qcommerce_orders_%)
                      qcommerce_orders_2015
                      qcommerce_orders_2016
                      qcommerce_orders_2017
                      qcommerce_orders_2018
                      qcommerce_orders_2019
                      qcommerce_orders_2020
                      qcommerce_orders_2021
                      qcommerce_orders_2022
                      qcommerce_orders_2023
                      qcommerce_orders_2024
                      qcommerce_orders_2025
                       qcommerce_orders_all

Total rows via view: 10,000,000


### Querying a single year directly
> Because each year has its own table, you can target one year without touching the others — useful for year-specific debugging or fast iteration.

In [5]:
# Demo: query an INDIVIDUAL year's table directly (not the combined view)
# Every year is its own table, so you can inspect any single year in isolation

df_2020_only = run_query("SELECT app, COUNT(*) AS orders FROM qcommerce_orders_2020 GROUP BY app ORDER BY orders DESC")
print("2020 table queried directly (no UNION, no other years involved):")
df_2020_only

2020 table queried directly (no UNION, no other years involved):


,app,orders
0,Dunzo Daily,247491
1,Amazon Fresh,165262
2,Swiggy Instamart,137247


In [6]:
# Check year-wise row count 
# This confirms all 11 years are correctly loaded
sql = '''
    SELECT
        year,
        COUNT(*) AS orders,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM qcommerce_orders_all
    GROUP BY year
    ORDER BY year
'''
df_check = run_query(sql)
print("Year-wise data check:")
df_check

Year-wise data check:


,year,orders,first_date,last_date
0,2015,100000,2015-01-01,2015-12-31
1,2016,150000,2016-01-01,2016-12-31
2,2017,200000,2017-01-01,2017-12-31
3,2018,250000,2018-01-01,2018-12-31
4,2019,350000,2019-01-01,2019-12-31
5,2020,550000,2020-01-01,2020-12-31
6,2021,1000000,2021-01-01,2021-12-31
7,2022,1400000,2022-01-01,2022-12-31
8,2023,1600000,2023-01-01,2023-12-31
9,2024,1900000,2024-01-01,2024-12-31


In [7]:
# Validate app launch years
# Blinkit should NOT appear before 2021
# Amazon Fresh should NOT appear before 2019
# This proves our data is correctly calibrated

sql = '''
    SELECT
        app,
        MIN(year) AS first_year_in_data,
        COUNT(*)  AS total_orders
    FROM qcommerce_orders_all
    GROUP BY app
    ORDER BY first_year_in_data
'''
df_apps = run_query(sql)
print("App launch year validation:")
df_apps

App launch year validation:


,app,first_year_in_data,total_orders
0,Dunzo Daily,2015,1548135
1,Amazon Fresh,2019,495469
2,Swiggy Instamart,2020,2019369
3,BigBasket BB Now,2021,327320
4,Zepto,2021,2033241
5,Blinkit,2021,3467280
6,Flipkart Minutes,2024,109186


## Phase 3 : Basic Queries (Q01–Q10)
> **Concepts:** SELECT · COUNT · SUM · AVG · GROUP BY · ORDER BY · WHERE · HAVING
> **Business focus:** Revenue, order volumes, city performance, cancellations

In [8]:
# Q01 — Total orders and revenue per app (all years)
# Business question: Which app has the highest GMV?
# Concept: GROUP BY + COUNT + SUM + percentage

sql_q01 = '''
    SELECT
        app,
        COUNT(order_id)                                          AS total_orders,
        ROUND(SUM(final_amount_rs), 0)                          AS total_revenue_rs,
        ROUND(AVG(order_amount_rs), 2)                          AS avg_order_value_rs,
        ROUND(COUNT(order_id) * 100.0 / SUM(COUNT(order_id))
              OVER(), 2)                                        AS share_pct
    FROM qcommerce_orders_all
    WHERE order_status = 'Delivered'
    GROUP BY app
    ORDER BY total_revenue_rs DESC
'''

df_q01 = run_query(sql_q01)
print("Q01 — Revenue by App:")
df_q01

Q01 — Revenue by App:


,app,total_orders,total_revenue_rs,avg_order_value_rs,share_pct
0,Blinkit,3171081,2.128555e+09,733.60,34.72
1,Zepto,1859220,1.248094e+09,733.60,20.35
2,Swiggy Instamart,1846301,1.192535e+09,706.01,20.21
3,Dunzo Daily,1407148,6.319164e+08,490.64,15.41
4,Amazon Fresh,451347,2.341664e+08,566.87,4.94
5,BigBasket BB Now,299048,1.934677e+08,707.04,3.27
6,Flipkart Minutes,99956,7.087908e+07,775.95,1.09


In [9]:
# Save result to CSV (for outputs/ folder)
os.makedirs("outputs", exist_ok=True)
df_q01.to_csv("outputs/q01_revenue_by_app.csv", index=False)
print("Saved to outputs/q01_revenue_by_app.csv")

Saved to outputs/q01_revenue_by_app.csv


In [10]:
# Q02 — Year-wise growth in orders and revenue
# Business question: How fast did Q-Commerce grow?
# Concept: GROUP BY year + SUM + AVG

sql_q02 = '''
    SELECT
        year,
        COUNT(order_id)                AS total_orders,
        ROUND(SUM(final_amount_rs), 0) AS total_gmv_rs,
        ROUND(AVG(order_amount_rs), 2) AS avg_aov_rs,
        ROUND(AVG(delivery_time_min), 1) AS avg_delivery_min
    FROM qcommerce_orders_all
    WHERE order_status = 'Delivered'
    GROUP BY year
    ORDER BY year
'''

df_q02 = run_query(sql_q02)
print("Q02 — Yearly Growth:")
df_q02.to_csv("outputs/q02_yearly_growth.csv", index=False)
df_q02

Q02 — Yearly Growth:


,year,total_orders,total_gmv_rs,avg_aov_rs,avg_delivery_min
0,2015,90544,2.223365e+07,268.26,40.2
1,2016,135932,4.088811e+07,328.56,38.5
2,2017,181222,6.432618e+07,387.72,36.7
3,2018,226904,9.251590e+07,445.52,34.8
4,2019,318244,1.466828e+08,503.66,32.9
5,2020,500598,2.578955e+08,562.95,31.1
6,2021,911989,5.134261e+08,615.20,29.2
7,2022,1277265,7.748778e+08,662.87,27.4
8,2023,1461942,9.429799e+08,704.98,25.5
9,2024,1738857,1.188343e+09,746.89,23.9


In [11]:
# Q03 — Top 10 cities by total orders
# Business question: Which cities drive the most volume?
# Concept: ORDER BY + LIMIT

sql_q03 = '''
    SELECT
        city,
        city_tier,
        COUNT(order_id)                AS total_orders,
        ROUND(SUM(final_amount_rs), 0) AS total_revenue_rs,
        ROUND(AVG(order_amount_rs), 2) AS avg_aov_rs
    FROM qcommerce_orders_all
    WHERE order_status = 'Delivered'
    GROUP BY city, city_tier
    ORDER BY total_orders DESC
    LIMIT 10
'''

df_q03 = run_query(sql_q03)
print("Q03 — Top 10 Cities:")
df_q03.to_csv("outputs/q03_top_cities.csv", index=False)
df_q03

Q03 — Top 10 Cities:


,city,city_tier,total_orders,total_revenue_rs,avg_aov_rs
0,Delhi,Tier 1,846850,541042407.0,698.23
1,Mumbai,Tier 1,846787,542124366.0,699.66
2,Bengaluru,Tier 1,846658,541166201.0,698.41
3,Pune,Tier 1,846281,540759891.0,698.34
4,Kolkata,Tier 1,845540,541092131.0,699.34
5,Chennai,Tier 1,845328,541408531.0,699.79
6,Hyderabad,Tier 1,845141,540543343.0,699.09
7,Ahmedabad,Tier 1,844987,540053799.0,698.39
8,Surat,Tier 2,258545,155278272.0,656.41
9,Chandigarh,Tier 2,257596,155131290.0,658.31


In [12]:
# Q04 — Cancellation rate per app
# Business question: Which app has worst customer experience?
# Concept: CASE WHEN inside COUNT + percentage

sql_q04 = '''
    SELECT
        app,
        COUNT(order_id)                                               AS total_orders,
        SUM(CASE WHEN order_status = 'Cancelled' THEN 1 ELSE 0 END)  AS cancelled,
        SUM(CASE WHEN order_status = 'Returned'  THEN 1 ELSE 0 END)  AS returned,
        ROUND(
            SUM(CASE WHEN order_status = 'Cancelled' THEN 1 ELSE 0 END)
            * 100.0 / COUNT(order_id), 2
        )                                                             AS cancel_rate_pct
    FROM qcommerce_orders_all
    GROUP BY app
    ORDER BY cancel_rate_pct DESC
'''

df_q04 = run_query(sql_q04)
print("Q04 — Cancellation Rate by App:")
df_q04.to_csv("outputs/q04_cancel_rate.csv", index=False)
df_q04

Q04 — Cancellation Rate by App:


,app,total_orders,cancelled,returned,cancel_rate_pct
0,Dunzo Daily,1548135,112181.0,28806.0,7.25
1,Amazon Fresh,495469,34943.0,9179.0,7.05
2,BigBasket BB Now,327320,22145.0,6127.0,6.77
3,Swiggy Instamart,2019369,135682.0,37386.0,6.72
4,Blinkit,3467280,231603.0,64596.0,6.68
5,Zepto,2033241,135715.0,38306.0,6.67
6,Flipkart Minutes,109186,7235.0,1995.0,6.63


In [13]:
# Q05 — Average delivery time by city tier
# Business question: Are Tier 1 cities significantly faster?
# Concept: WHERE filter + GROUP BY + AVG

sql_q05 = '''
    SELECT
        city_tier,
        ROUND(AVG(delivery_time_min), 1)  AS avg_delivery_min,
        ROUND(MIN(delivery_time_min), 1)  AS fastest_delivery_min,
        ROUND(MAX(delivery_time_min), 1)  AS slowest_delivery_min,
        COUNT(order_id)                   AS delivered_orders
    FROM qcommerce_orders_all
    WHERE order_status = 'Delivered'
    GROUP BY city_tier
    ORDER BY avg_delivery_min
'''

df_q05 = run_query(sql_q05)
print("Q05 — Delivery Time by City Tier:")
df_q05.to_csv("outputs/q05_delivery_by_tier.csv", index=False)
df_q05

Q05 — Delivery Time by City Tier:


,city_tier,avg_delivery_min,fastest_delivery_min,slowest_delivery_min,delivered_orders
0,Tier 3,23.7,5.0,134.9,567042
1,Tier 2,24.4,5.0,128.1,1799487
2,Tier 1,27.1,5.0,136.7,6767572


In [14]:
# Q06 — Payment mode share over years
# Business question: How did UPI adoption grow year on year?
# Concept: Window function for % within each year

sql_q06 = '''
    SELECT
        year,
        payment_mode,
        COUNT(order_id)  AS orders,
        ROUND(
            COUNT(order_id) * 100.0
            / SUM(COUNT(order_id)) OVER (PARTITION BY year),
            1
        ) AS pct_of_year
    FROM qcommerce_orders_all
    GROUP BY year, payment_mode
    ORDER BY year, orders DESC
'''

df_q06 = run_query(sql_q06)
print("Q06 — Payment Mode Evolution:")
df_q06.to_csv("outputs/q06_payment_mode.csv", index=False)
df_q06.head(20)

Q06 — Payment Mode Evolution:


,year,payment_mode,orders,pct_of_year
0,2015,Cash on Delivery,81952,82.0
1,2015,Wallet,7055,7.1
2,2015,Debit Card,6975,7.0
3,2015,Credit Card,2972,3.0
4,2015,Buy Now Pay Later,1046,1.0
5,2016,Cash on Delivery,109300,72.9
6,2016,Wallet,18189,12.1
7,2016,Debit Card,11979,8.0
8,2016,Credit Card,5938,4.0
9,2016,Buy Now Pay Later,3117,2.1


In [15]:
# Q07 — Category performance: volume vs revenue
# Business question: Which category should get more dark store space?
# Concept: Multi-metric GROUP BY + sorting

sql_q07 = '''
    SELECT
        category,
        COUNT(order_id)                AS total_orders,
        ROUND(SUM(final_amount_rs), 0) AS total_revenue_rs,
        ROUND(AVG(order_amount_rs), 2) AS avg_order_value_rs,
        ROUND(AVG(items_ordered), 1)   AS avg_items_per_order
    FROM qcommerce_orders_all
    WHERE order_status = 'Delivered'
    GROUP BY category
    ORDER BY total_revenue_rs DESC
'''

df_q07 = run_query(sql_q07)
print("Q07 — Category Performance:")
df_q07.to_csv("outputs/q07_category_performance.csv", index=False)
df_q07

Q07 — Category Performance:


,category,total_orders,total_revenue_rs,avg_order_value_rs,avg_items_per_order
0,Electronics,912785,579857120.0,694.24,3.5
1,Groceries & Staples,914877,570052559.0,681.00,3.5
2,Beauty & Skincare,914994,569856558.0,680.46,3.5
3,Personal Care,913404,569003826.0,680.82,3.5
4,Fruits & Vegetables,912565,568940763.0,681.30,3.5
5,Household,913995,568897723.0,680.38,3.5
6,Baby Products,913785,568740289.0,680.30,3.5
7,Medicines,913409,568269793.0,680.08,3.5
8,Dairy & Eggs,911876,568242735.0,680.87,3.5
9,Snacks & Beverages,912411,567752217.0,680.05,3.5


In [16]:
# Q08 — Festive vs Normal period comparison
# Business question: How much does Diwali boost business?
# Concept: CASE WHEN flag → GROUP BY

sql_q08 = '''
    SELECT
        year,
        CASE WHEN is_festive_season = 1
             THEN 'Festive Season'
             ELSE 'Normal Period'
        END                            AS period_type,
        COUNT(order_id)                AS total_orders,
        ROUND(AVG(order_amount_rs), 2) AS avg_aov_rs,
        ROUND(AVG(discount_rs), 2)     AS avg_discount_rs
    FROM qcommerce_orders_all
    GROUP BY year, is_festive_season
    ORDER BY year, is_festive_season DESC
'''

df_q08 = run_query(sql_q08)
print("Q08 — Festive vs Normal:")
df_q08.to_csv("outputs/q08_festive_analysis.csv", index=False)
df_q08

Q08 — Festive vs Normal:


,year,period_type,total_orders,avg_aov_rs,avg_discount_rs
0,2015,Festive Season,524,742.79,63.79
1,2015,Normal Period,99476,266.10,23.20
2,2016,Festive Season,810,825.39,73.76
3,2016,Normal Period,149190,326.04,28.44
4,2017,Festive Season,1123,938.14,75.06
5,2017,Normal Period,198877,384.75,33.52
6,2018,Festive Season,1348,964.19,79.73
7,2018,Normal Period,248652,442.67,38.75
8,2019,Festive Season,1902,1071.73,90.94
9,2019,Normal Period,348098,500.71,43.86


In [17]:
# Q09 — Hour-wise order distribution
# Business question: What time of day is busiest?
# Concept: GROUP BY hour + time analysis

sql_q09 = '''
    SELECT
        hour,
        COUNT(order_id)                  AS total_orders,
        ROUND(AVG(delivery_time_min), 1) AS avg_del_min,
        ROUND(AVG(order_amount_rs), 2)   AS avg_aov_rs
    FROM qcommerce_orders_all
    WHERE order_status = 'Delivered'
    GROUP BY hour
    ORDER BY hour
'''

df_q09 = run_query(sql_q09)
print("Q09 — Hourly Pattern:")
df_q09.to_csv("outputs/q09_hourly_pattern.csv", index=False)
df_q09

Q09 — Hourly Pattern:


,hour,total_orders,avg_del_min,avg_aov_rs
0,0,34654,26.6,679.71
1,1,17339,26.5,682.75
2,2,8688,27.1,682.49
3,3,8633,26.6,682.06
4,4,17330,26.5,689.30
5,5,43170,26.4,677.89
6,6,129466,26.4,681.12
7,7,301086,26.4,683.41
8,8,501123,26.4,681.65
9,9,645650,26.4,681.68


In [18]:
# Q10 — Tier 2 and Tier 3 city growth story
# Business question: When did Q-Commerce reach smaller cities?
# Concept: WHERE filter + year grouping

sql_q10 = '''
    SELECT
        year,
        city_tier,
        COUNT(order_id)  AS orders,
        ROUND(
            COUNT(order_id) * 100.0
            / SUM(COUNT(order_id)) OVER (PARTITION BY year),
            1
        ) AS pct_of_year
    FROM qcommerce_orders_all
    GROUP BY year, city_tier
    ORDER BY year, city_tier
'''

df_q10 = run_query(sql_q10)
print("Q10 — City Tier Expansion:")
df_q10.to_csv("outputs/q10_tier_expansion.csv", index=False)
df_q10

Q10 — City Tier Expansion:


,year,city_tier,orders,pct_of_year
0,2015,Tier 1,100000,100.0
1,2016,Tier 1,150000,100.0
2,2017,Tier 1,197988,99.0
3,2017,Tier 2,2012,1.0
4,2018,Tier 1,242667,97.1
5,2018,Tier 2,7333,2.9
6,2019,Tier 1,336013,96.0
7,2019,Tier 2,13987,4.0
8,2020,Tier 1,522756,95.0
9,2020,Tier 2,27244,5.0


## Phase 4 : Intermediate Queries (Q11–Q20)
> **Concepts:** CTE (WITH) · LAG() · CASE WHEN · Subqueries · ROW_NUMBER
> **Business focus:** Growth rates, market share, trends, rankings

In [19]:
# Q11 — Year-on-Year growth rate per app
# Business question: Which app grew fastest in 2021?
# Concept: CTE + LAG() window function

sql_q11 = '''
    WITH yearly_orders AS (
        SELECT
            app,
            year,
            COUNT(order_id) AS orders
        FROM qcommerce_orders_all
        GROUP BY app, year
    ),
    with_prev_year AS (
        SELECT
            app,
            year,
            orders,
            LAG(orders) OVER (
                PARTITION BY app
                ORDER BY year
            ) AS prev_year_orders
        FROM yearly_orders
    )
    SELECT
        app,
        year,
        orders                 AS current_orders,
        prev_year_orders,
        ROUND(
            (orders - prev_year_orders) * 100.0 / prev_year_orders,
            1
        )                      AS yoy_growth_pct
    FROM with_prev_year
    WHERE prev_year_orders IS NOT NULL
    ORDER BY year, yoy_growth_pct DESC
'''

df_q11 = run_query(sql_q11)
print("Q11 — YoY Growth by App:")
df_q11.to_csv("outputs/q11_yoy_growth.csv", index=False)
df_q11.head(20)

Q11 — YoY Growth by App:


,app,year,current_orders,prev_year_orders,yoy_growth_pct
0,Dunzo Daily,2016,150000,100000,50.0
1,Dunzo Daily,2017,200000,150000,33.3
2,Dunzo Daily,2018,250000,200000,25.0
3,Dunzo Daily,2019,192504,250000,-23.0
4,Dunzo Daily,2020,247491,192504,28.6
5,Amazon Fresh,2020,165262,157496,4.9
6,Swiggy Instamart,2021,239916,137247,74.8
7,Dunzo Daily,2021,280101,247491,13.2
8,Amazon Fresh,2021,130461,165262,-21.1
9,Zepto,2022,307702,79714,286.0


In [20]:
# Q12 — Market share battle 
# Business question: Which app gained in 2021?
# Concept: CTE + JOIN + percentage

sql_q12 = '''
    WITH yearly_total AS (
        SELECT year, COUNT(order_id) AS year_total
        FROM qcommerce_orders_all
        GROUP BY year
    ),
    yearly_app AS (
        SELECT year, app, COUNT(order_id) AS app_orders
        FROM qcommerce_orders
        GROUP BY year, app
    )
    SELECT
        ya.year,
        ya.app,
        ya.app_orders,
        ROUND(ya.app_orders * 100.0 / yt.year_total, 2) AS market_share_pct
    FROM yearly_app ya
    JOIN yearly_total yt ON ya.year = yt.year
    ORDER BY ya.year, market_share_pct DESC
'''

df_q12 = run_query(sql_q12)
print("Q12 — Market Share by Year:")
df_q12.to_csv("outputs/q12_market_share.csv", index=False)
df_q12.head(25)

Q12 — Market Share by Year:


,year,app,app_orders,market_share_pct
0,2015,Dunzo Daily,100000,100.00
1,2016,Dunzo Daily,150000,100.00
2,2017,Dunzo Daily,200000,100.00
3,2018,Dunzo Daily,250000,100.00
4,2019,Dunzo Daily,192504,55.00
5,2019,Amazon Fresh,157496,45.00
6,2020,Dunzo Daily,247491,45.00
7,2020,Amazon Fresh,165262,30.05
8,2020,Swiggy Instamart,137247,24.95
9,2021,Dunzo Daily,280101,28.01


In [21]:
# Q13 — Monthly orders + 3-month rolling average
# Business question: Is growth steady or volatile?
# Concept: AVG() OVER with ROWS BETWEEN frame

sql_q13 = '''
    WITH monthly_orders AS (
        SELECT
            year,
            month,
            COUNT(order_id)  AS orders
        FROM qcommerce_orders_all
        GROUP BY year, month
    )
    SELECT
        year,
        month,
        orders,
        ROUND(
            AVG(orders) OVER (
                ORDER BY year, month
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ), 0
        ) AS rolling_3m_avg
    FROM monthly_orders
    ORDER BY year, month
'''

df_q13 = run_query(sql_q13)
print("Q13 — Monthly Orders + Rolling Average:")
df_q13.to_csv("outputs/q13_rolling_avg.csv", index=False)
df_q13.head(24)

Q13 — Monthly Orders + Rolling Average:


,year,month,orders,rolling_3m_avg
0,2015,1,8397,8397.0
1,2015,2,8351,8374.0
2,2015,3,8492,8413.0
3,2015,4,8226,8356.0
4,2015,5,8274,8331.0
5,2015,6,8244,8248.0
6,2015,7,8214,8244.0
7,2015,8,8237,8232.0
8,2015,9,8441,8297.0
9,2015,10,8456,8378.0


In [22]:
# Q14 — Top 3 apps in each city tier in 2024
# Concept: ROW_NUMBER() to rank within group

sql_q14 = '''
    SELECT city_tier, app, total_orders, order_rank
    FROM (
        SELECT
            city_tier,
            app,
            COUNT(order_id) AS total_orders,
            ROW_NUMBER() OVER (
                PARTITION BY city_tier
                ORDER BY COUNT(order_id) DESC
            ) AS order_rank
        FROM qcommerce_orders_all
        WHERE year = 2024
        GROUP BY city_tier, app
    ) ranked
    WHERE order_rank <= 3
    ORDER BY city_tier, order_rank
'''

df_q14 = run_query(sql_q14)
print("Q14 — Top 3 Apps per City Tier in 2024:")
df_q14.to_csv("outputs/q14_top3_apps_tier.csv", index=False)
df_q14

Q14 — Top 3 Apps per City Tier in 2024:


,city_tier,app,total_orders,order_rank
0,Tier 1,Blinkit,547914,1
1,Tier 1,Zepto,345688,2
2,Tier 1,Swiggy Instamart,263207,3
3,Tier 2,Blinkit,231158,1
4,Tier 2,Zepto,145997,2
5,Tier 2,Swiggy Instamart,110900,3
6,Tier 3,Blinkit,77158,1
7,Tier 3,Zepto,48499,2
8,Tier 3,Swiggy Instamart,36839,3


In [23]:
# Q15 — AOV trend: which app is rising vs falling?
# Concept: CTE + LAG() + CASE WHEN label

sql_q15 = '''
    WITH app_aov AS (
        SELECT
            app,
            year,
            ROUND(AVG(order_amount_rs), 2) AS avg_aov
        FROM qcommerce_orders_all
        WHERE order_status = 'Delivered'
        GROUP BY app, year
    ),
    aov_with_prev AS (
        SELECT
            app, year, avg_aov,
            LAG(avg_aov) OVER (
                PARTITION BY app ORDER BY year
            ) AS prev_aov
        FROM app_aov
    )
    SELECT
        app, year, avg_aov, prev_aov,
        ROUND(avg_aov - prev_aov, 2) AS aov_change,
        CASE
            WHEN avg_aov > prev_aov THEN '↑ Rising'
            WHEN avg_aov < prev_aov THEN '↓ Falling'
            ELSE '→ Flat'
        END                          AS aov_trend
    FROM aov_with_prev
    WHERE prev_aov IS NOT NULL
    ORDER BY app, year
'''

df_q15 = run_query(sql_q15)
print("Q15 — AOV Trend per App:")
df_q15.to_csv("outputs/q15_aov_trend.csv", index=False)
df_q15

Q15 — AOV Trend per App:


,app,year,avg_aov,prev_aov,aov_change,aov_trend
0,Amazon Fresh,2020,563.71,502.95,60.76,↑ Rising
1,Amazon Fresh,2021,616.36,563.71,52.65,↑ Rising
2,Amazon Fresh,2022,663.70,616.36,47.34,↑ Rising
3,BigBasket BB Now,2022,661.73,615.12,46.61,↑ Rising
4,BigBasket BB Now,2023,704.73,661.73,43.00,↑ Rising
5,BigBasket BB Now,2024,749.73,704.73,45.00,↑ Rising
6,BigBasket BB Now,2025,791.32,749.73,41.59,↑ Rising
7,Blinkit,2022,663.71,615.58,48.13,↑ Rising
8,Blinkit,2023,704.71,663.71,41.00,↑ Rising
9,Blinkit,2024,747.22,704.71,42.51,↑ Rising


In [24]:
# Q16 — Top 5 Apps by City Tier (2024)
# Business question: Which are the top 5 quick-commerce apps in each city tier based on delivered orders in 2024?
# Concept: CTE + GROUP BY + DENSE_RANK() Window Function

sql_q16 = '''
    WITH app_orders AS (
        SELECT
            city_tier,
            app,
            COUNT(order_id) AS total_orders
        FROM qcommerce_orders_all
        WHERE year = 2024
          AND order_status = 'Delivered'
        GROUP BY city_tier, app
    ),
    ranked_apps AS (
        SELECT
            city_tier,
            app,
            total_orders,
            DENSE_RANK() OVER (
                PARTITION BY city_tier
                ORDER BY total_orders DESC
            ) AS app_rank
        FROM app_orders
    )

    SELECT
        city_tier,
        app,
        total_orders,
        app_rank
    FROM ranked_apps
    WHERE app_rank <= 5
    ORDER BY city_tier, app_rank
'''

df_q16 = run_query(sql_q16)
print("Q16 — Top 5 Apps by City Tier (2024):")
df_q16.to_csv("outputs/q16_citytierwise_app.csv", index=False)
df_q16

Q16 — Top 5 Apps by City Tier (2024):


,city_tier,app,total_orders,app_rank
0,Tier 1,Blinkit,501007,1
1,Tier 1,Zepto,316413,2
2,Tier 1,Swiggy Instamart,241189,3
3,Tier 1,BigBasket BB Now,32726,4
4,Tier 1,Flipkart Minutes,21785,5
5,Tier 2,Blinkit,211564,1
6,Tier 2,Zepto,133568,2
7,Tier 2,Swiggy Instamart,101581,3
8,Tier 2,BigBasket BB Now,13677,4
9,Tier 2,Flipkart Minutes,8937,5


In [25]:
# Q17 — Weekend vs Weekday Performance by Year
# Business question: How do weekend and weekday orders compare in terms of order volume and average order value across different years?
# Concept: CASE WHEN + GROUP BY + COUNT + AVG

sql_q17 = '''
    SELECT
        year,
        CASE
            WHEN is_weekend = 1 THEN 'Weekend'
            ELSE 'Weekday'
        END AS day_type,
        COUNT(order_id) AS total_orders,
        ROUND(AVG(order_amount_rs), 2) AS avg_order_value_rs
    FROM qcommerce_orders_all
    GROUP BY year, is_weekend
    ORDER BY year, day_type
'''

df_q17 = run_query(sql_q17)
print("Q17 — Weekend vs Weekday Performance by Year:")
df_q17.to_csv("outputs/q17_weekend_weekday_performance.csv", index=False)
df_q17

Q17 — Weekend vs Weekday Performance by Year:


,year,day_type,total_orders,avg_order_value_rs
0,2015,Weekday,71351,269.27
1,2015,Weekend,28649,266.92
2,2016,Weekday,107011,327.66
3,2016,Weekend,42989,331.44
4,2017,Weekday,142365,389.06
5,2017,Weekend,57635,384.88
6,2018,Weekday,178571,446.38
7,2018,Weekend,71429,443.22
8,2019,Weekday,249894,502.99
9,2019,Weekend,100106,505.85


In [26]:
# Q18 — Discount Impact on Order Value
# Business question: How do different discount ranges influence order volume, average order value, and delivery time?
# Concept: CASE WHEN + Bucket Analysis + GROUP BY + AVG

sql_q18 = '''
    SELECT
        CASE
            WHEN discount_rs = 0 THEN 'No Discount'
            WHEN discount_rs <= 50 THEN '₹1–50 Discount'
            WHEN discount_rs <= 150 THEN '₹51–150 Discount'
            ELSE '₹150+ Discount'
        END AS discount_bucket,
        COUNT(order_id) AS total_orders,
        ROUND(AVG(order_amount_rs), 2) AS avg_gross_order_value_rs,
        ROUND(AVG(final_amount_rs), 2) AS avg_net_order_value_rs,
        ROUND(AVG(delivery_time_min), 1) AS avg_delivery_time_min
    FROM qcommerce_orders_all
    WHERE order_status = 'Delivered'
    GROUP BY discount_bucket
    ORDER BY avg_gross_order_value_rs
'''

df_q18 = run_query(sql_q18)
print("Q18 — Discount Impact on Order Value:")
df_q18.to_csv("outputs/q18_discount_impact.csv", index=False)
df_q18

Q18 — Discount Impact on Order Value:


,discount_bucket,total_orders,avg_gross_order_value_rs,avg_net_order_value_rs,avg_delivery_time_min
0,₹1–50 Discount,1283218,465.41,433.32,27.6
1,₹51–150 Discount,2460929,638.03,545.42,26.3
2,No Discount,3984857,681.64,681.64,26.4
3,₹150+ Discount,1405097,957.49,772.25,25.4


In [27]:
# Q19 — Monthly Orders with 3-Month Rolling Average
# Business question: How have monthly order volumes evolved over time, and what is the 3-month rolling average trend?
# Concept: CTE + GROUP BY + Window Function (ROWS BETWEEN)

sql_q19 = '''
    WITH monthly_orders AS (
        SELECT
            year,
            month,
            COUNT(order_id) AS total_orders
        FROM qcommerce_orders_all
        GROUP BY year, month
    )

    SELECT
        year,
        month,
        total_orders,
        ROUND(
            AVG(total_orders) OVER (
                ORDER BY year, month
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ),
            0
        ) AS rolling_3m_avg_orders
    FROM monthly_orders
    ORDER BY year, month
'''

df_q19 = run_query(sql_q19)
print("Q19 — Monthly Orders with 3-Month Rolling Average:")
df_q19.to_csv("outputs/q19_orderwise_rolling_avg.csv", index=False)
df_q19

Q19 — Monthly Orders with 3-Month Rolling Average:


,year,month,total_orders,rolling_3m_avg_orders
0,2015,1,8397,8397.0
1,2015,2,8351,8374.0
2,2015,3,8492,8413.0
3,2015,4,8226,8356.0
4,2015,5,8274,8331.0
...,...,...,...,...
127,2025,8,207714,208241.0
128,2025,9,208704,208183.0
129,2025,10,209268,208562.0
130,2025,11,208426,208799.0


In [28]:
# Q20 — Fastest Growing Categories (Year-over-Year)
# Business question: Which product categories are experiencing the highest year-over-year growth in delivered orders?
# Concept: CTE + LAG() Window Function + YoY Growth Calculation

sql_q20 = '''
    WITH cat_yearly AS (
        SELECT
            category,
            year,
            COUNT(order_id) AS total_orders
        FROM qcommerce_orders_all
        WHERE order_status = 'Delivered'
        GROUP BY category, year
    ),
    cat_growth AS (
        SELECT
            category,
            year,
            total_orders,
            LAG(total_orders) OVER (
                PARTITION BY category
                ORDER BY year
            ) AS previous_year_orders
        FROM cat_yearly
    )

    SELECT
        category,
        year,
        total_orders,
        ROUND(
            (total_orders - previous_year_orders) * 100.0 /
            previous_year_orders,
            1
        ) AS yoy_growth_pct
    FROM cat_growth
    WHERE previous_year_orders IS NOT NULL
    ORDER BY year, yoy_growth_pct DESC
'''

df_q20 = run_query(sql_q20)
print("Q20 — Fastest Growing Categories (Year-over-Year):")
df_q20.to_csv("outputs/q20_categories_trend_yearwise.csv", index=False)
df_q20

Q20 — Fastest Growing Categories (Year-over-Year):


,category,year,total_orders,yoy_growth_pct
0,Baby Products,2016,13599,53.6
1,Beauty & Skincare,2016,13728,52.0
2,Snacks & Beverages,2016,13757,51.5
3,Medicines,2016,13683,51.2
4,Personal Care,2016,13711,50.7
...,...,...,...,...
95,Beauty & Skincare,2025,229520,31.6
96,Personal Care,2025,228328,31.5
97,Baby Products,2025,228982,31.4
98,Medicines,2025,228912,31.4


## Phase 5 : Advanced Queries (Q16–Q30)
> **Concepts:** RANK · DENSE_RANK · NTILE · Running totals · Cohort analysis
> **Business focus:** Performance ranking, city segmentation, Dunzo collapse proof

In [29]:
# Q21 — Rank apps by revenue within each year
# Concept: RANK() window function

sql_q21 = '''
    SELECT
        year,
        app,
        ROUND(SUM(final_amount_rs), 0)  AS revenue_rs,
        RANK() OVER (
            PARTITION BY year
            ORDER BY SUM(final_amount_rs) DESC
        )                               AS revenue_rank
    FROM qcommerce_orders_all
    WHERE order_status = 'Delivered'
    GROUP BY year, app
    ORDER BY year, revenue_rank
'''

df_q21 = run_query(sql_q21)
print("Q21 — App Revenue Ranking by Year:")
df_q21.to_csv("outputs/q21_revenue_rank.csv", index=False)
df_q21.head(20)

Q21 — App Revenue Ranking by Year:


,year,app,revenue_rs,revenue_rank
0,2015,Dunzo Daily,22233646.0,1
1,2016,Dunzo Daily,40888108.0,1
2,2017,Dunzo Daily,64326182.0,1
3,2018,Dunzo Daily,92515901.0,1
4,2019,Dunzo Daily,80755504.0,1
5,2019,Amazon Fresh,65927276.0,2
6,2020,Dunzo Daily,115888075.0,1
7,2020,Amazon Fresh,77657201.0,2
8,2020,Swiggy Instamart,64350210.0,3
9,2021,Dunzo Daily,143792996.0,1


In [30]:
# Q22 — Running total GMV across all years
# Concept: SUM() OVER cumulative window

sql_q22 = '''
    WITH monthly AS (
        SELECT
            year, month,
            SUM(final_amount_rs) AS monthly_gmv
        FROM qcommerce_orders_all
        WHERE order_status = 'Delivered'
        GROUP BY year, month
    )
    SELECT
        year, month,
        ROUND(monthly_gmv, 0)  AS monthly_gmv_rs,
        ROUND(
            SUM(monthly_gmv) OVER (
                ORDER BY year, month
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ), 0
        )                      AS cumulative_gmv_rs
    FROM monthly
    ORDER BY year, month
'''

df_q22 = run_query(sql_q22)
print("Q22 — Running Total GMV:")
df_q22.to_csv("outputs/q22_running_gmv.csv", index=False)
df_q22.head(20)

Q22 — Running Total GMV:


,year,month,monthly_gmv_rs,cumulative_gmv_rs
0,2015,1,1847483.0,1847483.0
1,2015,2,1844729.0,3692213.0
2,2015,3,1878040.0,5570252.0
3,2015,4,1822556.0,7392808.0
4,2015,5,1832955.0,9225763.0
5,2015,6,1802988.0,11028751.0
6,2015,7,1809920.0,12838672.0
7,2015,8,1831711.0,14670382.0
8,2015,9,1855936.0,16526318.0
9,2015,10,1875951.0,18402270.0


In [31]:
# Q23 — NTILE: split cities into revenue quartiles
# Concept: NTILE(4) — divide rows into 4 equal groups

sql_q23 = '''
    WITH city_revenue AS (
        SELECT
            city,
            city_tier,
            ROUND(SUM(final_amount_rs), 0) AS total_revenue
        FROM qcommerce_orders_all
        WHERE order_status = 'Delivered'
        GROUP BY city, city_tier
    )
    SELECT
        city, city_tier, total_revenue,
        NTILE(4) OVER (ORDER BY total_revenue DESC) AS quartile,
        CASE NTILE(4) OVER (ORDER BY total_revenue DESC)
            WHEN 1 THEN 'Q1 — Top 25 percent'
            WHEN 2 THEN 'Q2 — Upper Mid 25 percent'
            WHEN 3 THEN 'Q3 — Lower Mid 25 percent'
            WHEN 4 THEN 'Q4 — Bottom 25 percent'
        END                                          AS revenue_segment
    FROM city_revenue
    ORDER BY quartile, total_revenue DESC
'''

df_q23 = run_query(sql_q23)
print("Q23 — City Revenue Quartiles:")
df_q23.to_csv("outputs/q23_city_quartiles.csv", index=False)
df_q23.head(15)

Q23 — City Revenue Quartiles:


,city,city_tier,total_revenue,quartile,revenue_segment
0,Mumbai,Tier 1,542124366.0,1,Q1 — Top 25 percent
1,Chennai,Tier 1,541408531.0,1,Q1 — Top 25 percent
2,Bengaluru,Tier 1,541166201.0,1,Q1 — Top 25 percent
3,Kolkata,Tier 1,541092131.0,1,Q1 — Top 25 percent
4,Delhi,Tier 1,541042407.0,1,Q1 — Top 25 percent
5,Pune,Tier 1,540759891.0,2,Q2 — Upper Mid 25 percent
6,Hyderabad,Tier 1,540543343.0,2,Q2 — Upper Mid 25 percent
7,Ahmedabad,Tier 1,540053799.0,2,Q2 — Upper Mid 25 percent
8,Surat,Tier 2,155278272.0,2,Q2 — Upper Mid 25 percent
9,Indore,Tier 2,155265783.0,2,Q2 — Upper Mid 25 percent


In [32]:
# Q24 — Delivery SLA compliance rate by app
# Business: What % of orders delivered in under 20 min?
# Concept: CASE WHEN + percentage per group

sql_q24 = '''
    SELECT
        year,
        app,
        COUNT(order_id)    AS delivered_orders,
        SUM(CASE WHEN delivery_time_min <= 20
                 THEN 1 ELSE 0 END)  AS within_20_min,
        ROUND(
            SUM(CASE WHEN delivery_time_min <= 20
                     THEN 1 ELSE 0 END) * 100.0
            / COUNT(order_id), 1
        )                  AS sla_compliance_pct
    FROM qcommerce_orders_all
    WHERE order_status = 'Delivered'
    GROUP BY year, app
    ORDER BY year, sla_compliance_pct DESC
'''

df_q24 = run_query(sql_q24)
print("Q24 — Delivery SLA Compliance:")
df_q24.to_csv("outputs/q24_sla_compliance.csv", index=False)
df_q24.head(20)

Q24 — Delivery SLA Compliance:


,year,app,delivered_orders,within_20_min,sla_compliance_pct
0,2015,Dunzo Daily,90544,8075.0,8.9
1,2016,Dunzo Daily,135932,18893.0,13.9
2,2017,Dunzo Daily,181222,34246.0,18.9
3,2018,Dunzo Daily,226904,54493.0,24.0
4,2019,Amazon Fresh,143200,41996.0,29.3
5,2019,Dunzo Daily,175044,50951.0,29.1
6,2020,Swiggy Instamart,124929,42876.0,34.3
7,2020,Amazon Fresh,150521,51578.0,34.3
8,2020,Dunzo Daily,225148,77237.0,34.3
9,2021,BigBasket BB Now,45519,18118.0,39.8


In [33]:
# Q25 — DENSE_RANK: consistently top 3 apps across all years
# Concept: DENSE_RANK + filter + aggregation

sql_q25 = '''
    WITH app_yearly_rank AS (
        SELECT
            year, app,
            COUNT(order_id) AS orders,
            DENSE_RANK() OVER (
                PARTITION BY year
                ORDER BY COUNT(order_id) DESC
            ) AS order_rank
        FROM qcommerce_orders_all
        GROUP BY year, app
    )
    SELECT
        app,
        COUNT(year)               AS years_in_top3,
        MIN(order_rank)           AS best_rank_ever,
        ROUND(AVG(order_rank), 1) AS avg_rank
    FROM app_yearly_rank
    WHERE order_rank <= 3
    GROUP BY app
    ORDER BY years_in_top3 DESC, avg_rank
'''

df_q25 = run_query(sql_q25)
print("Q25 — Consistently Top 3 Apps:")
df_q25.to_csv("outputs/q25_top3_consistent.csv", index=False)
df_q25

Q25 — Consistently Top 3 Apps:


,app,years_in_top3,best_rank_ever,avg_rank
0,Dunzo Daily,7,1,1.0
1,Swiggy Instamart,6,2,2.7
2,Blinkit,5,1,1.4
3,Zepto,4,2,2.3
4,Amazon Fresh,2,2,2.0


In [34]:
# Q26 — Dunzo collapse: quarterly order decline
# Business: Prove Dunzo's fall with data
# Concept: CASE WHEN quarter + GROUP BY

sql_q26 = '''
    SELECT
        year,
        CASE
            WHEN month IN (1,2,3)  THEN 'Q1'
            WHEN month IN (4,5,6)  THEN 'Q2'
            WHEN month IN (7,8,9)  THEN 'Q3'
            ELSE                        'Q4'
        END                              AS quarter,
        COUNT(order_id)                  AS dunzo_orders,
        ROUND(AVG(order_amount_rs), 2)   AS avg_aov_rs
    FROM qcommerce_orders_all
    WHERE app = 'Dunzo Daily'
    GROUP BY year, quarter
    ORDER BY year, quarter
'''

df_q26 = run_query(sql_q26)
print("Q26 — Dunzo Collapse (Quarterly):")
df_q26.to_csv("outputs/q26_dunzo_collapse.csv", index=False)
df_q26

Q26 — Dunzo Collapse (Quarterly):


,year,quarter,dunzo_orders,avg_aov_rs
0,2015,Q1,25240,266.08
1,2015,Q2,24744,266.22
2,2015,Q3,24892,266.18
3,2015,Q4,25124,275.87
4,2016,Q1,37597,326.51
5,2016,Q2,37314,325.20
6,2016,Q3,37658,326.21
7,2016,Q4,37431,337.06
8,2017,Q1,49961,386.04
9,2017,Q2,50058,384.47


In [35]:
# Q27 — App performance scorecard 2024
# Business: Full multi-metric scorecard for each app
# Concept: Multiple aggregations in one query

sql_q27 = '''
    SELECT
        app,
        COUNT(order_id)                                           AS total_orders,
        ROUND(SUM(final_amount_rs), 0)                           AS total_gmv_rs,
        ROUND(AVG(order_amount_rs), 2)                           AS avg_aov_rs,
        ROUND(AVG(delivery_time_min), 1)                         AS avg_del_min,
        ROUND(
            SUM(CASE WHEN order_status = 'Cancelled'
                     THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
        )                                                        AS cancel_rate_pct,
        ROUND(AVG(customer_rating), 2)                           AS avg_rating,
        ROUND(AVG(discount_rs), 2)                               AS avg_discount_rs
    FROM qcommerce_orders_all
    WHERE year = 2024
    GROUP BY app
    ORDER BY total_gmv_rs DESC
'''

df_q27 = run_query(sql_q27)
print("Q27 — App Scorecard 2024:")
df_q27.to_csv("outputs/q27_app_scorecard.csv", index=False)
df_q27

Q27 — App Scorecard 2024:


,app,total_orders,total_gmv_rs,avg_aov_rs,avg_del_min,cancel_rate_pct,avg_rating,avg_discount_rs
0,Blinkit,856230,585625712.0,747.39,24.1,6.65,3.99,65.39
1,Zepto,540184,368707515.0,746.03,24.1,6.62,3.99,65.41
2,Swiggy Instamart,410946,280899098.0,747.09,24.1,6.56,3.99,65.48
3,BigBasket BB Now,55646,38151840.0,749.47,24.1,6.52,3.99,65.88
4,Flipkart Minutes,36994,25319364.0,748.18,24.2,6.70,3.99,65.72


In [36]:
# Q28 — Bengaluru-only era proof (2015–2016)
# Business question: Was Dunzo Daily primarily concentrated in Bengaluru during its early years?
# Concept: GROUP BY + COUNT + Window Function (PARTITION BY) + Percentage

sql_q28 = '''
    SELECT
        year,
        city,
        COUNT(order_id) AS total_orders,
        ROUND(
            COUNT(order_id) * 100.0 /
            SUM(COUNT(order_id)) OVER (PARTITION BY year),
            2
        ) AS pct_of_year
    FROM qcommerce_orders_all
    WHERE year IN (2015, 2016)
      AND app = 'Dunzo Daily'
    GROUP BY year, city
    ORDER BY year, total_orders DESC
'''

df_q28 = run_query(sql_q28)
print("Q28 — Bengaluru-only era proof (2015–2016):")
df_q28.to_csv("outputs/q28_dunzo_app_earlystage.csv", index=False)
df_q28

Q28 — Bengaluru-only era proof (2015–2016):


,year,city,total_orders,pct_of_year
0,2015,Pune,12607,12.61
1,2015,Ahmedabad,12562,12.56
2,2015,Bengaluru,12555,12.56
3,2015,Delhi,12522,12.52
4,2015,Hyderabad,12504,12.50
5,2015,Kolkata,12497,12.50
6,2015,Mumbai,12485,12.49
7,2015,Chennai,12268,12.27
8,2016,Kolkata,18955,12.64
9,2016,Chennai,18828,12.55


In [37]:
# Q29 — Peak Hour Revenue Contribution (Running %)
# Business question: How much revenue does each hour contribute, and how does cumulative revenue build throughout the day?
# Concept: CTE + SUM + Window Function (Running Total) + Percentage

sql_q29 = '''
    WITH hourly AS (
        SELECT
            hour,
            SUM(final_amount_rs) AS revenue
        FROM qcommerce_orders_all
        WHERE order_status = 'Delivered'
        GROUP BY hour
    ),
    total AS (
        SELECT
            SUM(final_amount_rs) AS grand_total
        FROM qcommerce_orders_all
        WHERE order_status = 'Delivered'
    )

    SELECT
        h.hour,
        ROUND(h.revenue, 0) AS hourly_revenue_rs,
        ROUND(h.revenue * 100.0 / t.grand_total, 2) AS pct_of_total_revenue,
        ROUND(
            SUM(h.revenue) OVER (
                ORDER BY h.hour
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) * 100.0 / t.grand_total,
            2
        ) AS running_pct
    FROM hourly h
    CROSS JOIN total t
    ORDER BY h.hour
'''

df_q29 = run_query(sql_q29)
print("Q29 — Peak Hour Revenue Contribution (Running %):")
df_q29.to_csv("outputs/q29_hourwise_revenue_share.csv", index=False)
df_q29

Q29 — Peak Hour Revenue Contribution (Running %):


,hour,hourly_revenue_rs,pct_of_total_revenue,running_pct
0,0,21577833.0,0.38,0.38
1,1,10847938.0,0.19,0.57
2,2,5440756.0,0.10,0.66
3,3,5383013.0,0.09,0.76
4,4,10941697.0,0.19,0.95
5,5,26794322.0,0.47,1.42
6,6,80644235.0,1.41,2.84
7,7,188246738.0,3.30,6.14
8,8,312610694.0,5.48,11.62
9,9,402761726.0,7.07,18.69


In [38]:
# Q30 — COVID Lockdown Spike Ratio (2020)
# Business question: How did order volume change during the COVID lockdown compared to the pre- and post-lockdown periods?
# Concept: CASE WHEN + GROUP BY + COUNT + AVG (per month)

sql_q30 = '''
    SELECT
        CASE
            WHEN month IN (3, 4, 5, 6) THEN 'Lockdown (Mar-Jun)'
            WHEN month IN (1, 2) THEN 'Pre-lockdown (Jan-Feb)'
            ELSE 'Post-lockdown (Jul-Dec)'
        END AS period,
        COUNT(order_id) AS total_orders,
        ROUND(
            COUNT(order_id) * 1.0 / COUNT(DISTINCT month),
            0
        ) AS avg_orders_per_month
    FROM qcommerce_orders_all
    WHERE year = 2020
    GROUP BY period
    ORDER BY
        CASE period
            WHEN 'Pre-lockdown (Jan-Feb)' THEN 1
            WHEN 'Lockdown (Mar-Jun)' THEN 2
            ELSE 3
        END
'''

df_q30 = run_query(sql_q30)
print("Q30 — COVID Lockdown Spike Ratio (2020):")
df_q30.to_csv("outputs/q30_covid_scorecard.csv", index=False)
df_q30

Q30 — COVID Lockdown Spike Ratio (2020):


,period,total_orders,avg_orders_per_month
0,Pre-lockdown (Jan-Feb),55747,27874.0
1,Lockdown (Mar-Jun),244449,61112.0
2,Post-lockdown (Jul-Dec),249804,41634.0


## Phase 6 : Validation & RFM Queries (Q31–Q33)
> These use the newer columns added to the schema: `customer_lifetime_orders`, `days_since_last_order`, `promo_code_used`, `estimated_delivery_distance_km`, and the MNAR flag columns.

In [39]:
# Q31 — RFM-Style User Segmentation
# Business question: How do different user types compare in terms of order frequency, recency, and average order value?
# Concept: GROUP BY + AVG + ORDER BY

sql_q31 = '''
    SELECT
        user_type,
        ROUND(AVG(customer_lifetime_orders), 1) AS avg_lifetime_orders,
        ROUND(AVG(days_since_last_order), 1) AS avg_days_since_last_order,
        ROUND(AVG(order_amount_rs), 2) AS avg_order_value_rs
    FROM qcommerce_orders_all
    GROUP BY user_type
    ORDER BY avg_lifetime_orders DESC
'''

df_q31 = run_query(sql_q31)
print("Q31 — RFM-Style User Segmentation:")
df_q31.to_csv("outputs/q31_rfm_user_segmentation.csv", index=False)
df_q31

Q31 — RFM-Style User Segmentation:


,user_type,avg_lifetime_orders,avg_days_since_last_order,avg_order_value_rs
0,Premium User,115.0,7.5,699.04
1,Returning User,22.0,30.5,689.78
2,New User,NaN,NaN,634.16


In [40]:
# Q32 — Promo Code Effectiveness
# Business question: How do promo code users and non-users compare in terms of order volume, average order value, and cancellation rate?
# Concept: GROUP BY + COUNT + AVG + CASE WHEN + Percentage

sql_q32 = '''
    SELECT
        promo_code_used,
        COUNT(order_id) AS total_orders,
        ROUND(AVG(order_amount_rs), 2) AS avg_order_value_rs,
        ROUND(
            SUM(CASE WHEN order_status = 'Cancelled' THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*),
            2
        ) AS cancellation_rate_pct
    FROM qcommerce_orders_all
    GROUP BY promo_code_used
    ORDER BY total_orders DESC
'''

df_q32 = run_query(sql_q32)
print("Q32 — Promo Code Effectiveness:")
df_q32.to_csv("outputs/q32_promo_code_scorecard.csv", index=False)
df_q32

Q32 — Promo Code Effectiveness:


,promo_code_used,total_orders,avg_order_value_rs,cancellation_rate_pct
0,No Promo,5976464,671.69,6.81
1,Festival Offer,700127,683.38,6.78
2,Referral,568173,681.77,6.78
3,Cashback Offer,554897,690.68,6.78
4,None,500472,681.55,6.81
5,Influencer Code,485995,716.80,6.81
6,Loyalty Code,484901,723.58,6.71
7,Bank Offer,452269,698.27,6.76
8,UPI Cashback,214793,773.39,6.50
9,COVID Special,46818,563.55,7.13


## Phase 7 : Key SQL Findings

### Top 10 Business Insights from SQL Analysis

| Query | Finding |
|-------|---------|
| Q01 | Blinkit generates highest GMV despite not being oldest app |
| Q02 | 2021 was biggest year — 3 new apps entered simultaneously |
| Q03 | Mumbai + Delhi + Bengaluru = top 3 cities consistently |
| Q04 | Dunzo had highest cancellation rate in its later years |
| Q05 | Tier 1 cities deliver 40% faster than Tier 3 cities |
| Q06 | UPI grew from 30% (2015) → 52% (2024) of payments |
| Q11 | 2021 saw 200%+ YoY growth for Blinkit, Zepto, BB Now |
| Q12 | Amazon Fresh market share fell from 45% (2019) → <5% (2024) |
| Q22 | Cumulative GMV crossed ₹1 crore milestone in 2021 |
| Q26 | Dunzo orders declined every single quarter from 2022 onwards |

---

### SQL Concepts Covered in This Notebook

| Concept | Query | What It Does |
|---------|-------|-------------|
| `GROUP BY` | Q01–Q10 | Aggregates rows into groups |
| `COUNT / SUM / AVG` | All | Basic aggregation functions |
| `CASE WHEN` | Q04, Q08 | Conditional logic inside SQL |
| `LIMIT` | Q03 | Returns only top N rows |
| `WITH` (CTE) | Q11–Q15 | Breaks complex query into steps |
| `LAG()` | Q11, Q15 | Gets value from previous row |
| `ROW_NUMBER()` | Q14 | Assigns unique number per group |
| `RANK()` | Q21 | Ranks with gaps for ties |
| `DENSE_RANK()` | Q25 | Ranks without gaps for ties |
| `NTILE(4)` | Q23 | Divides rows into N equal buckets |
| `SUM() OVER` cumulative | Q22 | Running total |
| `AVG() OVER ROWS` | Q13 | Rolling average |
---
*Project by Shubham Patil | Q-Commerce Dark Store SQL Analytics*
